# Tiled & Batched Grad-CAM Adversarial-Attack **Detection** (2xT4 ready)

This builds on `gradcam_attack_localization.ipynb`. That notebook handled **one image at a
time** and used **Grad-CAM**, which needs a *backward pass per image* -- fine for 10 animals,
a disaster for thousands of images or for video FPS.

**The idea here (your plan):** cut every image into a grid of small **tiles**, push *all tiles
of all images* through the GPU as **one big batch**, and ask "was *this tile* attacked?" for
each piece independently. Tiling gives spatial precision; batching keeps the GPU saturated.

**Why gradient-free CAM?** Inspired by the paper you linked -- *ViT-ReciproCAM*
([arXiv:2310.02588](https://arxiv.org/abs/2310.02588)) -- which shows you can get CAM saliency
**without gradients or attention**. Gradient-free CAM (here **EigenCAM** from
[`jacobgil/pytorch-grad-cam`](https://github.com/jacobgil/pytorch-grad-cam)) is the key to
batching: no per-sample backprop means a whole stack of tiles resolves in a **single forward
pass**, and it scales trivially across **2 GPUs**.

### What this notebook does
1. Install `grad-cam` + deps.
2. Load **ResNet50** (ImageNet) and place a copy on **both T4 GPUs**.
3. **Tile** an image into an NxN grid; classify + CAM every tile in **one batched forward**.
4. **Blind per-tile attack detector** (high-frequency energy -- needs *no* clean reference).
5. **FGSM** attack confined to chosen tiles -> ground truth of which tiles are adversarial.
6. **Benchmark throughput**: images/sec & tiles/sec, **1-GPU vs 2-GPU**, batched vs one-at-a-time.

> **Honesty about "speed":** tiling *increases* total pixels processed (16 tiles vs 1 image).
> The real wins are: (a) **gradient-free + batched** beats per-image backprop, (b) tiles keep
> **both GPUs fully fed**, and (c) you get **per-region localization for free**. The benchmark
> cell measures the actual numbers so you can see the tradeoff, not just trust the claim.

## 1. Install
`grad-cam` is the PyPI name of the `jacobgil/pytorch-grad-cam` library (import as
`pytorch_grad_cam`). On Kaggle, enable **Internet** and set the accelerator to **GPU T4 x2**.

In [ ]:
!pip install -q grad-cam einops tqdm

In [ ]:
# ============================================================
# CELL 1 - Setup & GPU inventory
# ============================================================
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
import matplotlib.patches as patches
from torchvision import models, transforms
from PIL import Image
import urllib.request, json, os, time, math
from concurrent.futures import ThreadPoolExecutor

N_GPU = torch.cuda.device_count()
DEVICES = [f"cuda:{i}" for i in range(N_GPU)] if N_GPU else ["cpu"]
print("GPUs found:", N_GPU)
for i in range(N_GPU):
    print(f"  cuda:{i} -> {torch.cuda.get_device_name(i)}")
print("Using devices:", DEVICES)

## 2. Model on every GPU
For multi-GPU we keep **one independent copy of ResNet50 per device** (instead of
`DataParallel`). That lets each GPU run its own gradient-free CAM engine on its own slice of the
tile-batch, in parallel threads -- simpler and more reliable than wrapping the CAM hooks in
`DataParallel`.

In [ ]:
# ============================================================
# CELL 2 - One ResNet50 (ImageNet) per GPU + readable labels
# ============================================================
def make_model(dev):
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    return m.eval().to(dev)

MODELS = {d: make_model(d) for d in DEVICES}          # device -> model copy
print(f"Loaded {len(MODELS)} ResNet50 copies (one per device).")

# Human-readable ImageNet class names
url = "https://raw.githubusercontent.com/raghakot/keras-vis/master/resources/imagenet_class_index.json"
idx2label = {int(k): v[1] for k, v in json.load(urllib.request.urlopen(url)).items()}

## 3. Preprocessing + the **tiler**
Images live in `[0,1]` pixel space; ImageNet normalization happens *inside* the model call so the
adversarial noise stays measured in real pixels (same convention as your old notebook).

`tile_image` splits a `[3,S,S]` image into a `GRID x GRID` stack of tiles. ResNet is fully
convolutional + adaptive-pooled, so it classifies the smaller tiles directly (a `112x112` tile
costs ~1/4 of a `224x224` forward).

In [ ]:
# ============================================================
# CELL 3 - Preprocessing, tiling, helpers
# ============================================================
SIZE = 448          # working resolution per image
GRID = 4            # GRID x GRID tiles  (16 tiles/image)
TILE = SIZE // GRID # tile side in pixels (112)

to_tensor = transforms.Compose([transforms.Resize((SIZE, SIZE)), transforms.ToTensor()])

_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_std  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
def normalize(t):                       # normalize on the tensor's own device
    return (t - _mean.to(t.device)) / _std.to(t.device)

def load_image(path_or_url):
    # Load local path OR URL -> [1,3,SIZE,SIZE] float tensor in [0,1] (on CPU).
    if str(path_or_url).startswith("http"):
        fn = "/tmp/" + os.path.basename(path_or_url)
        if not os.path.exists(fn):
            urllib.request.urlretrieve(path_or_url, fn)
        path_or_url = fn
    img = Image.open(path_or_url).convert("RGB")
    return to_tensor(img).unsqueeze(0)

def tile_image(x):
    # [1,3,SIZE,SIZE] -> [GRID*GRID, 3, TILE, TILE] (row-major tile order).
    p = x.unfold(2, TILE, TILE).unfold(3, TILE, TILE)        # [1,3,GRID,GRID,TILE,TILE]
    p = p.permute(0, 2, 3, 1, 4, 5).reshape(-1, 3, TILE, TILE)
    return p.contiguous()

def untile(tiles):
    # [GRID*GRID, C, TILE, TILE] -> [C, SIZE, SIZE] mosaic (inverse of tile_image).
    C = tiles.shape[1]
    g = tiles.reshape(GRID, GRID, C, TILE, TILE).permute(2, 0, 3, 1, 4)
    return g.reshape(C, SIZE, SIZE)

def tile_grid_to_full(per_tile_scalar):
    # [GRID*GRID] per-tile values -> [SIZE,SIZE] blocky heatmap for overlay.
    g = per_tile_scalar.reshape(GRID, GRID)
    return np.kron(g, np.ones((TILE, TILE)))

def to_np(t):  return t.squeeze().detach().cpu().permute(1, 2, 0).numpy()
def norm01(a):
    a = np.asarray(a, np.float32)
    return (a - a.min()) / (a.max() - a.min() + 1e-8)

## 4. Gradient-free CAM, batched across both GPUs
We use **EigenCAM** from `pytorch_grad_cam`: it takes the first principal component of the target
layer's activations -- **no gradients, no class target, one forward pass**. We hold one `EigenCAM`
per GPU and, for a big tile-batch, split it and run both engines **concurrently in threads** (CUDA
ops release the GIL -> genuine 2-GPU parallelism).

In [ ]:
# ============================================================
# CELL 4 - Multi-GPU gradient-free CAM engine
# ============================================================
from pytorch_grad_cam import EigenCAM

# One CAM engine per device, each bound to its own model copy's last conv block.
CAMS = {d: EigenCAM(model=MODELS[d], target_layers=[MODELS[d].layer4[-1]]) for d in DEVICES}
for c in CAMS.values():
    c.batch_size = 64

def batched_cam(tiles_cpu, use_gpus=None):
    # tiles_cpu: [B,3,h,w] on CPU. Returns grayscale CAMs [B,h,w] (numpy), split across GPUs.
    devs = use_gpus or DEVICES
    if len(devs) == 1 or tiles_cpu.shape[0] < 2:
        d = devs[0]
        return CAMS[d](input_tensor=normalize(tiles_cpu).to(d), targets=None)
    chunks = torch.chunk(tiles_cpu, len(devs), dim=0)        # split batch across GPUs
    def _run(args):
        d, ch = args
        return CAMS[d](input_tensor=normalize(ch).to(d), targets=None)
    with ThreadPoolExecutor(max_workers=len(devs)) as ex:
        outs = list(ex.map(_run, zip(devs, chunks)))
    return np.concatenate(outs, axis=0)

@torch.no_grad()
def batched_predict(tiles_cpu, use_gpus=None):
    # Top-1 class id + confidence for each tile, split across GPUs.
    devs = use_gpus or DEVICES
    chunks = torch.chunk(tiles_cpu, len(devs), dim=0) if len(devs) > 1 else [tiles_cpu]
    devs = devs[:len(chunks)]
    def _run(args):
        d, ch = args
        p = MODELS[d](normalize(ch).to(d)).softmax(1)
        conf, idx = p.max(1)
        return idx.cpu().numpy(), conf.cpu().numpy()
    with ThreadPoolExecutor(max_workers=len(devs)) as ex:
        outs = list(ex.map(_run, zip(devs, chunks)))
    idx = np.concatenate([o[0] for o in outs]); conf = np.concatenate([o[1] for o in outs])
    return idx, conf

## 5. FGSM attack confined to chosen tiles
To get **ground truth** of "which tiles are adversarial," we run FGSM but zero the perturbation
outside a chosen set of tiles. `random_tile_mask` picks a few tiles to poison; the returned
`attacked_tiles` boolean vector is what our detector is graded against.

In [ ]:
# ============================================================
# CELL 5 - Per-tile FGSM attack (ground-truth generator)
# ============================================================
def tile_mask_from_ids(ids):
    # boolean [GRID*GRID] of attacked tiles -> pixel mask [1,1,SIZE,SIZE].
    m = torch.zeros(1, 1, SIZE, SIZE)
    for k in np.where(ids)[0]:
        r, c = divmod(int(k), GRID)
        m[..., r*TILE:(r+1)*TILE, c*TILE:(c+1)*TILE] = 1.0
    return m

def random_tile_mask(n_tiles=3, seed=None):
    rng = np.random.default_rng(seed)
    ids = np.zeros(GRID*GRID, bool)
    ids[rng.choice(GRID*GRID, size=n_tiles, replace=False)] = True
    return ids

def fgsm_attack(x, true_class, epsilon=0.06, tile_ids=None):
    # x:[1,3,SIZE,SIZE] in [0,1]. Confine noise to tile_ids if given. Returns adv image (CPU).
    d = DEVICES[0]
    xi = x.clone().detach().to(d).requires_grad_(True)
    out = MODELS[d](normalize(xi))
    loss = F.cross_entropy(out, torch.tensor([true_class], device=d))
    MODELS[d].zero_grad(); loss.backward()
    perturb = epsilon * xi.grad.sign()
    if tile_ids is not None:
        perturb = perturb * tile_mask_from_ids(tile_ids).to(d)
    return torch.clamp(xi + perturb, 0, 1).detach().cpu()

## 6. The blind per-tile detector
FGSM (and most pixel attacks) inject **high-frequency** noise. We score each tile by its
**high-frequency energy** = mean |tile - blur(tile)|. Adversarial tiles spike above the image's
own tile-population baseline, so we flag tiles where the score exceeds **median + k*MAD** across
that image's tiles. **No clean reference is needed** -- this works on images you didn't attack
yourself, which is the whole point of a deployable detector.

In [ ]:
# ============================================================
# CELL 6 - Blind high-frequency per-tile attack score (no clean ref needed)
# ============================================================
def _gaussian_kernel(sigma=1.0, ksize=5):
    ax = torch.arange(ksize) - ksize // 2
    g = torch.exp(-(ax**2) / (2*sigma**2)); g = g / g.sum()
    k = torch.outer(g, g)
    return k.view(1, 1, ksize, ksize).repeat(3, 1, 1, 1)   # depthwise, 3 ch

_GK = _gaussian_kernel()

def hf_energy_per_tile(tiles):
    # tiles:[B,3,h,w] in [0,1] -> [B] high-frequency energy score.
    k = _GK.to(tiles.device)
    blur = F.conv2d(tiles, k, padding=k.shape[-1]//2, groups=3)
    hf = (tiles - blur).abs().mean(dim=(1, 2, 3))
    return hf.cpu().numpy()

def detect_tiles(hf_scores, k=3.0):
    # Robust outlier flag: median + k*MAD. Returns boolean [B] flagged tiles + threshold.
    med = np.median(hf_scores)
    mad = np.median(np.abs(hf_scores - med)) + 1e-8
    thresh = med + k * 1.4826 * mad
    return hf_scores > thresh, thresh

def tile_metrics(true_ids, pred_ids):
    # Precision / recall / IoU at the tile level.
    tp = int((true_ids & pred_ids).sum())
    fp = int((~true_ids & pred_ids).sum())
    fn = int((true_ids & ~pred_ids).sum())
    prec = tp / (tp + fp) if tp + fp else float("nan")
    rec  = tp / (tp + fn) if tp + fn else float("nan")
    union = int((true_ids | pred_ids).sum())
    iou = tp / union if union else float("nan")
    return dict(precision=prec, recall=rec, tile_IoU=iou, tp=tp, fp=fp, fn=fn)

## 7. Full tiled pipeline for one image
Tile -> batch through both GPUs (CAM + prediction) -> blind detect -> compare to ground truth ->
visualize. Everything after the attack runs as **batched forward passes**.

In [ ]:
# ============================================================
# CELL 7 - One-image tiled pipeline + visualization
# ============================================================
def _overlay_tiles(ax, ids, color, ls="-"):
    for k in np.where(ids)[0]:
        r, c = divmod(int(k), GRID)
        ax.add_patch(patches.Rectangle((c*TILE, r*TILE), TILE, TILE,
                     fill=False, edgecolor=color, linewidth=2.5, linestyle=ls))

def run_one(path_or_url, n_attacked=3, epsilon=0.06, seed=0, show=True, name=""):
    x = load_image(path_or_url)                                  # [1,3,SIZE,SIZE] CPU

    # whole-image clean prediction (to choose the FGSM target class)
    cls0, conf0 = batched_predict(x)
    cls0 = int(cls0[0])

    # attack a few tiles -> ground truth
    attacked = random_tile_mask(n_attacked, seed=seed)
    x_adv = fgsm_attack(x, cls0, epsilon, attacked)

    # tile both clean & adversarial, batch through the GPUs
    t_clean, t_adv = tile_image(x), tile_image(x_adv)
    cam_clean = batched_cam(t_clean)                             # [T,h,w]
    cam_adv   = batched_cam(t_adv)
    pred_adv, conf_adv = batched_predict(t_adv)

    # blind detection on the adversarial tiles
    hf = hf_energy_per_tile(t_adv)
    flagged, thr = detect_tiles(hf)
    m = tile_metrics(attacked, flagged)

    # whole-image prediction after attack (did the global label flip?)
    clsA, confA = batched_predict(x_adv); clsA = int(clsA[0])
    res = dict(name=name or os.path.basename(str(path_or_url)),
               before=idx2label[cls0], after=idx2label[clsA],
               fooled=clsA != cls0, n_attacked=int(attacked.sum()),
               n_flagged=int(flagged.sum()), **m)

    if show:
        adv_np = to_np(x_adv)
        cam_shift = np.abs(cam_adv - cam_clean).mean(axis=(1, 2))     # per-tile CAM movement
        fig, ax = plt.subplots(1, 4, figsize=(18, 4.6))
        ax[0].imshow(adv_np)
        _overlay_tiles(ax[0], attacked, "lime");  _overlay_tiles(ax[0], flagged, "cyan", ls="--")
        ax[0].set_title("Adversarial image\nlime = TRUE attacked   cyan-- = DETECTED")
        ax[1].imshow(tile_grid_to_full(norm01(hf)), cmap="hot")
        ax[1].set_title("Blind HF-energy score\n(per tile)")
        ax[2].imshow(tile_grid_to_full(norm01(cam_shift)), cmap="jet")
        ax[2].set_title("CAM attention-shift\n|EigenCAM adv - clean|")
        cam_mosaic = untile(torch.tensor(cam_adv).unsqueeze(1).repeat(1, 3, 1, 1))
        ax[3].imshow(adv_np); ax[3].imshow(cam_mosaic[0], cmap="jet", alpha=0.5)
        ax[3].set_title("EigenCAM mosaic\n(all tiles, batched)")
        for a in ax: a.axis("off")
        flag = "FOOLED" if res["fooled"] else "label held"
        fig.suptitle(f"{res['name']}  --  {res['before']} -> {res['after']}  [{flag}]   "
                     f"tile-IoU={m['tile_IoU']:.2f}  P={m['precision']:.2f} R={m['recall']:.2f}",
                     y=1.03, fontsize=12)
        plt.tight_layout(); plt.show()
    return res

## Quick demo on one image
Lime boxes = tiles we actually attacked; dashed cyan = tiles the **blind** detector flagged. They
should line up, and the HF-energy panel should light up on the attacked tiles.

In [ ]:
_ = run_one(
    "https://raw.githubusercontent.com/EliSchwartz/imagenet-sample-images/master/n02129165_lion.JPEG",
    n_attacked=3, epsilon=0.08, seed=0)

## 8. Point it at a dataset
**Recommended Kaggle datasets** (accelerator **GPU T4 x2**, Internet **On**):

| Dataset | Kaggle slug / path | Why |
|---|---|---|
| **Imagenette** (10-class ImageNet subset) | search "imagenette" | Real ImageNet labels -> ResNet predicts meaningfully |
| **Animals-10** | `alessiocorrado99/animals10` -> `/kaggle/input/animals10/raw-img` | Matches your earlier animal demo |
| **ImageNet-1k val sample** | various mini-imagenet datasets | Largest, best for throughput numbers |

Set `FOLDER` to any image directory; otherwise it falls back to downloading a handful of sample
images (needs Internet).

In [ ]:
# ============================================================
# CELL 8 - Gather images (Kaggle dataset folder OR web fallback)
# ============================================================
CANDIDATE_FOLDERS = [
    "/kaggle/input/animals10/raw-img",
    "/kaggle/input/imagenette/imagenette2/val",
    "/kaggle/input/imagenette2/val",
]
FOLDER = next((f for f in CANDIDATE_FOLDERS if os.path.isdir(f)), "")

def gather_images(folder, n=20, seed=1):
    exts = (".jpg", ".jpeg", ".png", ".JPEG")
    paths = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.endswith(exts): paths.append(os.path.join(root, f))
    rng = np.random.default_rng(seed)
    return list(rng.choice(paths, size=min(n, len(paths)), replace=False)) if paths else []

if FOLDER:
    images = gather_images(FOLDER, n=20)
    print(f"Using {len(images)} images from {FOLDER}")
else:
    base = "https://raw.githubusercontent.com/EliSchwartz/imagenet-sample-images/master/"
    images = [base + n for n in [
        "n02099601_golden_retriever.JPEG", "n02123045_tabby.JPEG", "n02391049_zebra.JPEG",
        "n02129165_lion.JPEG", "n02129604_tiger.JPEG", "n02510455_giant_panda.JPEG",
        "n01518878_ostrich.JPEG", "n01806143_peacock.JPEG", "n01882714_koala.JPEG",
        "n02007558_flamingo.JPEG"]]
    print(f"No Kaggle folder found -- using {len(images)} downloaded samples")

## 9. Run detection over the set + summary table
Tile-level precision / recall / IoU averaged across images, plus the global fool-rate.

In [ ]:
# ============================================================
# CELL 9 - Batched detection over all images + summary
# ============================================================
import pandas as pd
results = [run_one(p, n_attacked=3, epsilon=0.08, seed=i, show=False)
           for i, p in enumerate(images)]
df = pd.DataFrame(results)
print(f"Global fool-rate: {df['fooled'].mean():.0%}")
print(f"Mean tile-IoU: {df['tile_IoU'].mean():.2f}   "
      f"Precision: {df['precision'].mean():.2f}   Recall: {df['recall'].mean():.2f}")
df[["name", "before", "after", "fooled", "n_attacked", "n_flagged",
    "precision", "recall", "tile_IoU"]]

## 10. Throughput benchmark -- the whole point
We measure **images/sec** and **tiles/sec** for the gradient-free batched CAM, on **1 GPU vs both
GPUs**, and contrast with a **one-image-at-a-time Grad-CAM** baseline (per-image backward pass,
like the original notebook). This is where you see whether tiling+batching actually pays off on
your hardware.

In [ ]:
# ============================================================
# CELL 10 - Throughput benchmark
# ============================================================
from pytorch_grad_cam import GradCAM

# Pre-load a working batch of tiles (reuse the first few images)
bench_imgs = [load_image(p) for p in images[:8]]
all_tiles = torch.cat([tile_image(x) for x in bench_imgs], dim=0)   # [8*16, 3, TILE, TILE]
print(f"Benchmark batch: {all_tiles.shape[0]} tiles ({len(bench_imgs)} images x {GRID*GRID})")

def time_it(fn, warmup=1, iters=3):
    for _ in range(warmup): fn()
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(iters): fn()
    if torch.cuda.is_available(): torch.cuda.synchronize()
    return (time.time() - t0) / iters

n_tiles = all_tiles.shape[0]
n_imgs = len(bench_imgs)

# (a) gradient-free EigenCAM, 1 GPU
t_1gpu = time_it(lambda: batched_cam(all_tiles, use_gpus=DEVICES[:1]))
# (b) gradient-free EigenCAM, all GPUs
t_all = time_it(lambda: batched_cam(all_tiles, use_gpus=DEVICES))
# (c) one-at-a-time gradient Grad-CAM baseline (per-tile backward), 1 GPU
_d = DEVICES[0]
_gcam = GradCAM(model=MODELS[_d], target_layers=[MODELS[_d].layer4[-1]])
def _onebyone():
    for i in range(n_tiles):
        _gcam(input_tensor=normalize(all_tiles[i:i+1]).to(_d), targets=None)
t_seq = time_it(_onebyone, warmup=0, iters=1)

print("\n=== Throughput (higher = better) ===")
for label, t in [("EigenCAM batched, 1 GPU", t_1gpu),
                 (f"EigenCAM batched, {len(DEVICES)} GPU", t_all),
                 ("Grad-CAM one-at-a-time, 1 GPU", t_seq)]:
    print(f"{label:34s}  {t*1000:7.1f} ms/batch   "
          f"{n_tiles/t:7.1f} tiles/s   {n_imgs/t:6.1f} img/s")
speedup = t_seq / t_all if t_all else float("nan")
print(f"\nBatched+{len(DEVICES)}GPU vs one-at-a-time speedup: {speedup:.1f}x")

## Recap & where to go next

**What we changed vs `gradcam_attack_localization.ipynb`:**
- Swapped per-image **Grad-CAM (backward pass)** -> gradient-free **EigenCAM (forward only)**,
  motivated by **ViT-ReciproCAM** ([arXiv:2310.02588](https://arxiv.org/abs/2310.02588)).
- **Tiled** each image and processed all tiles as **one batch**, **sharded across both T4 GPUs**.
- Added a **blind, reference-free per-tile attack detector** (high-frequency energy) and graded it
  with tile-level **precision / recall / IoU**.
- Added a **throughput benchmark** so the speed claim is measured, not assumed.

**Knobs:** `GRID` (tile granularity vs cost), `epsilon` (attack strength), `detect_tiles(k=...)`
(detector sensitivity), `SIZE` (working resolution).

**Next steps for the research log:**
- **True video FPS:** feed `cv2.VideoCapture` frames into `run_one` (drop the FGSM step, keep the
  blind detector) and report frames/sec.
- **Stronger attacks:** swap FGSM for PGD/AutoAttack to test detector robustness; HF-energy may
  need a learned tile classifier for low-frequency attacks.
- **Real ReciproCAM:** plug in the paper's ViT + ReciproCAM as the gradient-free backbone and
  compare saliency quality (the library exposes many CAMs to swap in).
- **DDP at scale:** for >2 GPUs or multi-node, replace the thread-split with
  `DistributedDataParallel`.